# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ameen740/Internship_Flyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_Token")

print("HF token loaded successfully!")

HF token loaded successfully!


In [5]:
!pip install -q duckdb

In [6]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_Token")

con = duckdb.connect()

con.execute(
    f"CREATE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}')"
)

rel = "hf://datasets/FlyRank/internship-warehouse"

print("Connected to FlyRank warehouse!")

Connected to FlyRank warehouse!


In [7]:
query = f"""
SELECT *
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
LIMIT 5
"""

con.sql(query)

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬───────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │ gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_clau

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

For Lane 2, one row represents one content item for one client on one report date. I will use the fact_content_daily_performance table and develop on the mid-panel month of March 2026 (month=2026-03). This provides a daily content-performance view while avoiding the final June 2026 month, which is reserved as a sealed test period.


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
query = f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

con.sql(query)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────┬────────────┬────────────┐
│ row_count │  min_date  │  max_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: Historical performance signals such as `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, and `ga4_sessions`. These are candidate features because they describe performance that has already been observed before the decision moment.

Label/proxy: The future decline outcome, which will be defined using a later observation window. Future outcome information will not be used as a feature.

Context: `client_hash_id`, `content_hash_id`, and `report_date` are used to identify, group, join, or split the data. They are context fields rather than predictive features.

Excluded: Label-derived fields such as `trend_pct` and `trend_direction` are excluded because they contain information derived from the outcome and could cause data leakage. Future-window information and fields that are unavailable at the decision moment are also excluded.


In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
grain_query = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS n
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
"""

con.sql(grain_query)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────┐
│ report_date │ client_hash_id │ content_hash_id │   n   │
│    date     │    varchar     │     varchar     │ int64 │
├─────────────┴────────────────┴─────────────────┴───────┤
│                         0 rows                         │
└────────────────────────────────────────────────────────┘

In [19]:
count_query = f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

con.sql(count_query)

┌───────────┬────────────┬────────────┐
│ row_count │  min_date  │  max_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘

In [21]:
missing_query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS NULL) AS gsc_availability_nulls,
    COUNT(*) FILTER (WHERE ga4_data_available IS NULL) AS ga4_availability_nulls,
    COUNT(*) FILTER (WHERE gsc_impressions IS NULL) AS gsc_impressions_nulls,
    COUNT(*) FILTER (WHERE gsc_clicks IS NULL) AS gsc_clicks_nulls,
    COUNT(*) FILTER (WHERE gsc_avg_position IS NULL) AS gsc_position_nulls,
    COUNT(*) FILTER (WHERE ga4_sessions IS NULL) AS ga4_sessions_nulls
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

con.sql(missing_query)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────────┬────────────────────────┬───────────────────────┬──────────────────┬────────────────────┬────────────────────┐
│ total_rows │ gsc_availability_nulls │ ga4_availability_nulls │ gsc_impressions_nulls │ gsc_clicks_nulls │ gsc_position_nulls │ ga4_sessions_nulls │
│   int64    │         int64          │         int64          │         int64         │      int64       │       int64        │       int64        │
├────────────┼────────────────────────┼────────────────────────┼───────────────────────┼──────────────────┼────────────────────┼────────────────────┤
│    9841378 │                      0 │                3018741 │                     0 │                0 │            6230317 │            3018741 │
└────────────┴────────────────────────┴────────────────────────┴───────────────────────┴──────────────────┴────────────────────┴────────────────────┘

In [22]:
window_query = f"""
SELECT
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(DISTINCT report_date) AS distinct_dates
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

con.sql(window_query)

┌────────────┬────────────┬────────────────┐
│  min_date  │  max_date  │ distinct_dates │
│    date    │    date    │     int64      │
├────────────┼────────────┼────────────────┤
│ 2026-03-01 │ 2026-03-31 │             31 │
└────────────┴────────────┴────────────────┘

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data has several limitations. Client history is unbalanced, so different clients have different amounts of available history. Some early rows are GSC-only because GA4 data was not yet available, and GA4 values in those rows can be zero-filled. Therefore, zeros cannot automatically be interpreted as zero engagement. The historical and future observation windows must also be kept separate to avoid overlap and data leakage. The data also cannot prove that a content change caused a performance change; it can only provide observed and measured signals for decision support.



In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.